In [12]:
import onnxruntime as ort
import numpy as np
import os
import torch
from omnivoice.utils.audio import (
    load_audio,
    remove_silence,
    trim_long_audio,
)
from IPython.display import Audio, display

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

# Test Dummy

### Encoder

In [3]:
sess_enc = ort.InferenceSession(
    os.path.join("onnx_higgs_audio_v2_tokenizer_encoder", "model.onnx"),
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)

In [4]:
dummy_audio = torch.randn(1, 1, 24000, device=device)

In [5]:
audio_np = dummy_audio.cpu().numpy().astype(np.float32)

In [6]:
audio_np.shape

(1, 1, 24000)

In [7]:
codes = sess_enc.run(None, {"input_values": audio_np})[0]

In [8]:
codes.shape

(1, 8, 25)

### Decoder

In [9]:
sess_dec = ort.InferenceSession(
    os.path.join("onnx_higgs_audio_v2_tokenizer_decoder", "model.onnx"),
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)

In [10]:
audio_out = sess_dec.run(None, {"audio_codes": codes})[0]

In [11]:
audio_out.shape

(1, 1, 24000)

# Test Real Voice Audio

### Encoder

In [13]:
sampling_rate = 24_000
ref_audio = "voice_audio.mp3"

In [14]:
ref_wav = load_audio(ref_audio, sampling_rate)

In [15]:
ref_wav

array([[0.        , 0.        , 0.        , ..., 0.20389651, 0.19312084,
        0.11391713]], shape=(1, 144000), dtype=float32)

In [16]:
display(Audio(ref_wav[0], rate=sampling_rate))

In [17]:
ref_wav.shape

(1, 144000)

In [18]:
audio_np = np.expand_dims(ref_wav, axis=0)

In [19]:
audio_np.shape

(1, 1, 144000)

In [20]:
codes = sess_enc.run(None, {"input_values": audio_np})[0]

In [21]:
codes.shape

(1, 8, 150)

In [22]:
codes

array([[[244, 244, 244, ..., 666, 312, 331],
        [503, 499, 499, ...,  71, 380, 901],
        [427, 977,  94, ..., 265, 179, 170],
        ...,
        [764, 153, 294, ..., 341, 694, 553],
        [926, 485, 944, ..., 342, 288, 360],
        [858, 588, 620, ..., 850, 826, 159]]],
      shape=(1, 8, 150), dtype=int64)

### Decoder

In [23]:
audio_out = sess_dec.run(None, {"audio_codes": codes})[0]

In [24]:
audio_out.shape

(1, 1, 144000)

In [25]:
display(Audio(audio_out[0][0], rate=sampling_rate))